# Domain Contrastive Fine-tuning — multilingual-e5-base

Goal: fine-tune `intfloat/multilingual-e5-base` with contrastive loss so that
articles from the same SR domain cluster tightly in embedding space.

Pipeline:
1. Load labeled articles (101k train, 5k gold test)
2. Build positive pairs per domain
3. Fine-tune with `MultipleNegativesRankingLoss`
4. Evaluate: within/cross-domain gap + classification accuracy on gold test
5. Classify unknown articles (69k) → assign domain labels
6. Save fine-tuned model + classified unknowns

## Section 1 — Setup

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')

BASE      = '../data'
MODEL_DIR = '../models/e5-domain-finetuned'
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

# ── Domain mapping (same as law-text-analysis notebook) ──────────────────────
ABBREV_TO_DOMAIN = {
    'BV':'1','AIG':'1','VZAE':'1','VG':'1','BGG':'1','PO':'1','BPV':'1',
    'EJPD':'1','EDA':'1','NDG':'1','ASG':'1','ISG':'1','VBP':'1','BGS':'1',
    'BPR':'1','RVOG':'1','RVOV':'1','GE':'1',
    'ZGB':'2','GBV':'2','GB':'2','ZV':'2','OR':'2','KG':'2','ZPO':'2',
    'ERV':'2','IPRG':'2','SV':'2','HG':'2','DS':'2','PF':'2','URG':'2',
    'VZG':'2','VVG':'2','BGBB':'2','DV':'2','BG':'2',
    'IRSG':'3',
    'ETH':'4','HFKG':'4','FIFG':'4','TSV':'4','BBV':'4',
    'MG':'5','MIG':'5','VMDP':'5','VVA':'5','VMSV':'5','WG':'5','BZG':'5','ZSV':'5','VBS':'5',
    'ZG':'6','MWSTG':'6','MWSTV':'6','DBG':'6','BAZG':'6','SVAV':'6',
    'SVG':'7','VRV':'7','SSV':'7','VZV':'7','VVV':'7','EBG':'7','EBV':'7',
    'VTS':'7','LFG':'7','LFV':'7','VIL':'7','SSG':'7','VPB':'7','FMG':'7',
    'FDV':'7','FMV':'7','PG':'7','VPG':'7','RTVG':'7','RTVV':'7','EMKV':'7',
    'KEG':'7','WV':'7','WRG':'7',
    'HMG':'8','VAM':'8','USG':'8','LMVV':'8','LGV':'8','VSFK':'8','ZDV':'8',
    'ZDG':'8','VUV':'8','AHVG':'8','AHVV':'8','IVG':'8','IVV':'8','BVG':'8',
    'KVG':'8','KVV':'8','KKV':'8','KLV':'8','UVG':'8','UVV':'8','AVIG':'8',
    'AVIV':'8','MVG':'8','LMG':'8','ATSG':'8','PBG':'8','KV':'8','AV':'8',
    'BSV':'8','VRAB':'8','PV':'8',
    'FINMA':'9','FINMAG':'9','AVO':'9','VAG':'9','KAG':'9','FV':'9',
    'FIDLEG':'9','FIDLEV':'9','FINIV':'9','UEV':'9','VGS':'9','SVV':'9',
    'DZV':'9','PSMV':'9',
}

DOMAIN_LABELS = {
    '1': '1xx: State & Constitution',    '2': '2xx: Private Law & Procedure',
    '3': '3xx: Criminal Law',            '4': '4xx: Education & Culture',
    '5': '5xx: Defense & Civil Protection', '6': '6xx: Finance & Tax',
    '7': '7xx: Transport & Telecom',     '8': '8xx: Health, Labor & Social',
    '9': '9xx: Economic Law',
}
DOMAIN_LIST = sorted(DOMAIN_LABELS.keys())
DOMAIN_TO_INT = {d: i for i, d in enumerate(DOMAIN_LIST)}  # '1'→0, '2'→1, ...

print('Setup complete')

## Section 2 — Load & Split Data

In [ ]:
import os

# ── Paths — works locally and on Kaggle ──────────────────────────────────────
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval'   # adjust to your dataset slug
else:
    DATA_DIR = '../data'

laws_df = pd.read_csv(f'{DATA_DIR}/laws_de.csv')
laws_df['text']  = laws_df['text'].fillna('')
laws_df['title'] = laws_df['title'].fillna('')
laws_df['law_code'] = laws_df['citation'].str.extract(r'(\d[\d\.]*\d|\d)\s*$')
laws_df['domain']   = laws_df['law_code'].str.extract(r'^(\d)', expand=False)

abbrev = laws_df.loc[laws_df['domain'].isna(), 'citation'].str.extract(r'([A-Z][A-Z0-9]+)\s*$')[0]
laws_df.loc[laws_df['domain'].isna(), 'domain'] = abbrev.map(ABBREV_TO_DOMAIN)
laws_df['domain'] = laws_df['domain'].fillna('?')

labeled_df = laws_df[laws_df['domain'] != '?'].copy().reset_index(drop=True)
unknown_df = laws_df[laws_df['domain'] == '?'].copy().reset_index(drop=True)

# ── Stratified 5k gold split — on the fly ────────────────────────────────────
rng_split = np.random.default_rng(42)
gold_idx  = []
for domain in DOMAIN_LIST:
    idx = labeled_df[labeled_df['domain'] == domain].index.tolist()
    k   = max(1, round(len(idx) / len(labeled_df) * 5000))
    gold_idx.extend(rng_split.choice(idx, size=min(k, len(idx)), replace=False).tolist())
gold_idx  = gold_idx[:5000]

gold_df   = labeled_df.loc[gold_idx].reset_index(drop=True)
train_df  = labeled_df.drop(index=gold_idx).reset_index(drop=True)

print(f'Train:   {len(train_df):,}  articles across {train_df["domain"].nunique()} domains')
print(f'Gold:    {len(gold_df):,}  articles (held-out test)')
print(f'Unknown: {len(unknown_df):,}  articles to classify')
print()
print('Train domain distribution:')
print(train_df['domain'].value_counts().sort_index().to_string())

## Section 3 — Build Training Pairs

For `MultipleNegativesRankingLoss`: each row is a positive pair `(anchor, positive)`
from the same domain. In-batch articles from other domains act as negatives automatically.

To keep it balanced, we sample the same number of pairs per domain.

In [ ]:
from datasets import Dataset

def build_input_text(title, text):
    """Prepend 'passage: ' prefix as required by E5 models."""
    title = str(title or '').strip()
    text  = str(text  or '').strip()
    combined = f"{title}. {text}" if title else text
    return 'passage: ' + combined[:512]

rng = np.random.default_rng(42)

PAIRS_PER_DOMAIN = 3000
anchors, positives = [], []

for domain in DOMAIN_LIST:
    df_d = train_df[train_df['domain'] == domain].reset_index(drop=True)
    n = len(df_d)
    if n < 2:
        continue
    idx_a = rng.choice(n, size=PAIRS_PER_DOMAIN, replace=True)
    idx_b = rng.choice(n, size=PAIRS_PER_DOMAIN, replace=True)
    mask  = idx_a != idx_b
    idx_a, idx_b = idx_a[mask], idx_b[mask]

    titles_a = df_d['title'].iloc[idx_a].tolist()
    texts_a  = df_d['text'].iloc[idx_a].tolist()
    titles_b = df_d['title'].iloc[idx_b].tolist()
    texts_b  = df_d['text'].iloc[idx_b].tolist()

    anchors.extend(  [build_input_text(t, x) for t, x in zip(titles_a, texts_a)])
    positives.extend([build_input_text(t, x) for t, x in zip(titles_b, texts_b)])

# Shuffle
order     = rng.permutation(len(anchors))
anchors   = [anchors[i]   for i in order]
positives = [positives[i] for i in order]

train_dataset = Dataset.from_dict({'anchor': anchors, 'positive': positives})

print(f'Total training pairs: {len(train_dataset):,}')
print(f'~{len(train_dataset)//len(DOMAIN_LIST):,} pairs per domain')
print(train_dataset)

## Section 4 — Fine-tune Model

In [ ]:
from sentence_transformers import SentenceTransformer, losses

MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
print(f'Loaded: {MODEL_NAME}')
print(f'Max sequence length: {model.max_seq_length}')

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample
import warnings
warnings.filterwarnings('ignore')   # suppress InputExample deprecation warning

# Convert Dataset back to InputExamples for model.fit()
train_examples = [
    InputExample(texts=[a, p])
    for a, p in zip(train_dataset['anchor'], train_dataset['positive'])
]

BATCH_SIZE   = 64
EPOCHS       = 3
WARMUP_RATIO = 0.1

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model)
warmup_steps     = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

print(f'Steps per epoch: {len(train_dataloader):,}')
print(f'Total steps:     {len(train_dataloader) * EPOCHS:,}')
print(f'Warmup steps:    {warmup_steps}')
print(f'In-batch negatives per pair: {BATCH_SIZE - 1}')

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=MODEL_DIR,
    show_progress_bar=True,
)
print(f'\nModel saved to {MODEL_DIR}')

## Section 5 — Evaluate on Gold Test Set

Two metrics:
1. **Within/cross-domain gap** — did fine-tuning push the gap above 0.02?
2. **kNN classification accuracy** — using embeddings, can we correctly predict domain?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

def encode_df(df, model, batch_size=64):
    texts = [build_input_text(t, x)
             for t, x in zip(df['title'].tolist(), df['text'].tolist())]
    return model.encode(texts, batch_size=batch_size,
                        show_progress_bar=True, normalize_embeddings=True)

print('Encoding gold test set...')
gold_embs  = encode_df(gold_df, model)
gold_labels = gold_df['domain'].tolist()

print('Encoding train sample (20k)...')
train_sample = train_df.sample(20_000, random_state=42)
train_embs   = encode_df(train_sample, model)
train_labels = train_sample['domain'].tolist()

In [ ]:
# ── Within vs cross-domain cosine similarity ─────────────────────────────────
SAMPLE_N = 1000
rng2 = np.random.default_rng(0)
idx  = rng2.choice(len(gold_embs), size=min(SAMPLE_N, len(gold_embs)), replace=False)
embs_s  = gold_embs[idx]
labels_s = np.array(gold_labels)[idx]

sim_matrix = embs_s @ embs_s.T
upper = np.triu_indices(len(idx), k=1)
sims  = sim_matrix[upper]
same  = labels_s[upper[0]] == labels_s[upper[1]]

print('=== Embedding space separation (gold test) ===')
print(f'Within-domain mean sim:  {sims[same].mean():.4f}')
print(f'Cross-domain  mean sim:  {sims[~same].mean():.4f}')
print(f'Gap:                     {sims[same].mean() - sims[~same].mean():.4f}')
print(f'  (baseline before fine-tuning was 0.0208)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sims[same],  bins=80, alpha=0.6, color='steelblue',  label='Same domain',  density=True)
ax.hist(sims[~same], bins=80, alpha=0.6, color='darkorange', label='Cross domain', density=True)
ax.set_xlabel('Cosine similarity'); ax.set_ylabel('Density')
ax.set_title('Within vs cross-domain similarity — fine-tuned model')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── kNN classification accuracy ──────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embs, train_labels)
pred_labels = knn.predict(gold_embs)

print('=== 5-NN Classification Report (gold test) ===')
print(classification_report(gold_labels, pred_labels,
                              target_names=[DOMAIN_LABELS[d] for d in sorted(set(gold_labels))],
                              digits=3))

## Section 6 — Classify Unknown Articles

In [ ]:
print(f'Encoding {len(unknown_df):,} unknown articles...')
unknown_embs = encode_df(unknown_df, model, batch_size=128)
unknown_pred  = knn.predict(unknown_embs)
unknown_proba = knn.predict_proba(unknown_embs).max(axis=1)

unknown_df = unknown_df.copy()
unknown_df['domain_pred'] = unknown_pred
unknown_df['domain_conf'] = unknown_proba

print('\nPredicted domain distribution for unknowns:')
print(unknown_df['domain_pred'].value_counts().sort_index().to_string())
print(f'\nMean confidence: {unknown_proba.mean():.3f}')
print(f'% with conf > 0.8: {(unknown_proba > 0.8).mean()*100:.1f}%')

In [ ]:
import os

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/input') else BASE
print(f'Saving to: {OUT_DIR}')

# Save classified unknowns
out_cols = ['citation', 'law_code', 'title', 'text', 'domain_pred', 'domain_conf']
unknown_df[out_cols].to_csv(f'{OUT_DIR}/unknown_classified.csv', index=False)
print(f'Saved unknown_classified.csv  ({len(unknown_df):,} rows)')

# Build full corpus with best-available domain label
full_df = laws_df.copy()
pred_map = unknown_df.set_index('citation')['domain_pred']
conf_map = unknown_df.set_index('citation')['domain_conf']

full_df.loc[full_df['domain'] == '?', 'domain'] = (
    full_df.loc[full_df['domain'] == '?', 'citation'].map(pred_map)
)
full_df['domain_source'] = 'citation'
full_df['domain_conf']   = 1.0
full_df.loc[full_df['citation'].isin(unknown_df['citation']), 'domain_source'] = 'model'
full_df.loc[full_df['citation'].isin(unknown_df['citation']), 'domain_conf']   = (
    full_df.loc[full_df['citation'].isin(unknown_df['citation']), 'citation'].map(conf_map)
)

full_df.to_csv(f'{OUT_DIR}/laws_de_with_domains.csv', index=False)
print(f'Saved laws_de_with_domains.csv ({len(full_df):,} rows)')

print('\nFinal domain distribution (full corpus):')
dist = full_df['domain'].value_counts().sort_index()
for d, n in dist.items():
    label = DOMAIN_LABELS.get(d, d)
    print(f'  {label:35s}  {n:6,}  ({100*n/len(full_df):.1f}%)')

still_unknown = (full_df['domain'] == '?').sum()
print(f'\nStill unclassified: {still_unknown:,}')